Code on RFE and Predictive maintenance.It predicts when a machine needs maintenance to prevent the machine from breaking down.
<br>Predictive maintenance is useful anywhere regular maintenance is required for a machine. 

# Objectives
  - One-hot encoding
  - Multivariate Imputation by Chained Equation (MICE)
  - SMOTE
  - Compare multiple algorithms
  - (RFE)

In [ ]:
import pandas as pd
import numpy as np
from sklearn import preprocessing, metrics
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.experimental import enable_iterative_imputer

In [ ]:
# Get Data
pmd_df = pd.read_csv('/kaggle/input/datasets/irabatibiswas/predict/ai4i2020m.csv')

In [ ]:
pmd_df.info()

In [ ]:
columns = pmd_df.columns.values.astype(str)
columns = np.char.replace(columns, old='[', new='', count=-1)
columns = np.char.replace(columns, old=']', new='', count=-1)
columns = np.char.replace(columns, old=' ', new='_', count=-1)
pmd_df.columns = columns
pmd_df.head(10)

In [ ]:

for col in pmd_df.columns:
   # print(pmd_df[col].value_counts())
    li = list(pmd_df[col].value_counts(dropna = False))
    print("No of unique value for col :", col ,"is (using shape)",pmd_df[col].value_counts().shape[0])
    print("Number of unique values in col :",col," is (using len) ", len(li))


In [ ]:
pmd_df_new=pmd_df.drop(columns=['UDI', 'Product_ID'])
pmd_df_new.head(5)

In [ ]:
# number of Missing Values as Question Marks in each colum
print('Question mark count for each col:')
qs_mark_counts = pmd_df_new.isin(['?']).sum()
print(qs_mark_counts)
# Replace ? with nan
print('Replace ? with nan:')
columns_to_replace = pmd_df_new.columns.difference(['Type'])
print('Cols to replace ',columns_to_replace)
for col_repl in columns_to_replace:
    pmd_df_new[col_repl]=pmd_df_new[col_repl].apply(pd.to_numeric, errors='coerce')
#Check for qs mark    
qs_mark_counts1 = pmd_df_new.isin(['?']).sum()
print(qs_mark_counts1)
# number of missing values in each colum
pmd_df_new.info()
print(' The size of the dataframe is:', pmd_df_new.shape)
# Show the distribution of nulls among the columns
print("###\nDistribution of nulls among columns:")
pmd_df_null_counts = pmd_df_new.isna().sum()
print(pmd_df_null_counts)
pmd_df_new.head(5)

In [ ]:
from sklearn.model_selection import train_test_split
pmd_df_new.info()
pmd_df_new.isna().sum(axis=0)
X = pmd_df_new.drop('Machine_failure', axis=1)
y = pmd_df_new['Machine_failure']
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.2, 
    stratify=y, 
    random_state=42
)

In [ ]:
from sklearn.preprocessing import OneHotEncoder
# Create a OneHotEncoder object
encoder = OneHotEncoder(sparse_output=False)
# Fit the OneHotEncoder object to the training data
encoder.fit(X_train[['Type']])
column_names = encoder.get_feature_names_out(['Type'])
# Encode the training data and the test data
# Add one-hot-encoded columns to the 2 dataframes
train_encoded = encoder.transform(X_train[['Type']])
test_encoded = encoder.transform(X_test[['Type']])
X_train[column_names] = train_encoded
X_test[column_names] = test_encoded
# Drop original categorical columns from the 2 dataframes
X_train = X_train.drop(columns=['Type'])
X_test = X_test.drop(columns=['Type'])
# Show first few rows
print(X_train.head())
print(X_test.head())
# Determine the number of (non-)missing values in the 2 dataframes
X_train.info()
X_test.info()

In [ ]:
# Import the imputer
# First, enable the experimental feature
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
# Create an IterativeImputer object
imputer = IterativeImputer(max_iter=10, random_state=0)
# Fit the IterativeImputer object
X_train_imputed = imputer.fit_transform(X_train)
# Transform the training data
X_train_imputed_df = pd.DataFrame(
    X_train_imputed,
    columns=X_train.columns,
    index=X_train.index
)

print("Training Data Information After Imputation:")
print(X_train_imputed_df.info())
print("\nMissing values per column in X_train:")
print(X_train_imputed_df.isna().sum())

X_test_imputed = imputer.transform(X_test)
X_test_imputed_df = pd.DataFrame(X_test_imputed, columns=X_test.columns)
print("Test Data Information After Imputation:")
print(X_test_imputed_df.info())
print("\nMissing values per column in X_test:")
print(X_test_imputed_df.isna().sum())

In [ ]:
!pip install imblearn #imbalanced-learn
from imblearn.over_sampling import SMOTE

In [ ]:
# Check original training dataset size
from collections import Counter
print(f"Original training shape: {X_train_imputed_df.shape}")

print(f"Original distribution: {Counter(y_train)}")
print(y_train.value_counts(normalize=True) * 100)
# Create a SMOTE object and resample the training data
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train_imputed_df, y_train)

print(f"Resampled training shape: {X_train_sm.shape}")
print(f"Resampled distribution: {Counter(y_train_sm)}")
print(y_train_sm.value_counts(normalize=True) * 100)



<br>based on the training data, and evaluate their performance on the test dataset. Use default hyperparameter values, except for `LogisticRegression(max_iter=1000)` and ` SVC(probability=True)`.  Setting `SVC(probability=False)` will not provide the required soft predictions (aka probabilities).
<br>  In order to use XGBClassifier you must have xgboost installed.  If you do not have xgboost installed, then run `!pip install xgboost`.


In [ ]:
!pip install xgboost
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_auc_score

In [ ]:

def PrettyConfusingMatrix(y_test, y_pred):
    conmat = pd.DataFrame(data=confusion_matrix(y_test, y_pred, labels=[0, 1]),
                          columns=['predicted no', 'predicted yes'], 
                          index=['actual no', 'actual yes'])
    display(conmat)

def BinaryClassificationAccuracy(y_test, y_pred_proba, threshold=0.5):
    y_pred = y_pred_proba > threshold
    PrettyConfusingMatrix(y_test, y_pred)
    print(classification_report(y_test, y_pred))
    if np.all(y_pred_proba == y_pred):
        print('\n cannot calculate AUC of ROC from hard predictions')
    else:   
        print('\nAUC of ROC:', np.round(roc_auc_score(y_true=y_test, y_score=y_pred_proba), 2))

In [ ]:

models = {
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "SVC": SVC(probability=True, random_state=42),
    "KNeighbors": KNeighborsClassifier(n_neighbors=5),
    "DecisionTree": DecisionTreeClassifier(random_state=42),
    "XGBoost": XGBClassifier(eval_metric='logloss', random_state=42)
}

for name, model in models.items():
    print(f"Model Train & predict:--- {name} ---")    
    model.fit(X_train_sm, y_train_sm)
    
    y_pred = model.predict(X_test_imputed_df)    
    # 3. Determine probabilities of machine failure (class 1)predict_proba returns [prob_class_0, prob_class_1]
    y_probs = model.predict_proba(X_test_imputed_df)[:, 1]
    print("Y probability :",y_probs)
    print(f"\nConfusion Matrix:")
    BinaryClassificationAccuracy(y_test,y_probs,threshold=0.5)
    auc = roc_auc_score(y_test, y_probs)
    print(f"AUC-ROC: {auc:.2f}\n")

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.metrics import recall_score, accuracy_score, precision_score
# Create logistic regression object
log_reg = LogisticRegression(max_iter=1000)
current_n = 8
final_rfe = None
while current_n > 0:
    # Initialize and fit RFE on SMOTE-augmented data (X_train_smote, y_train_smote)
     rfe = RFE(estimator=log_reg, n_features_to_select=current_n)
     rfe.fit(X_train_sm,y_train_sm)
     y_pred_rfe = rfe.predict(X_test_imputed_df)
     y_pred_prob_rfe = rfe.predict_proba(X_test_imputed_df)[:, 1]
     #print(y_pred_rfe)
     recall = recall_score(y_test, y_pred_rfe)
     print('Recall -',recall)
     if recall < 0.75:
        # Revert to the last successful model if current recall is too low
        break
     final_rfe = rfe
     current_n -= 1
# Calculate the accuracy measures
print('Current n :',current_n)
y_pred_final = final_rfe.predict_proba(X_test_imputed_df)[:, 1]
print(f"Final Features Selected: {final_rfe.n_features_}")
BinaryClassificationAccuracy(y_test,y_pred_final,threshold=0.50)
print("Feature Rankings:", final_rfe.ranking_)
selected_features = X_train_sm.columns[final_rfe.support_]
print("Selected Features:", selected_features.tolist())

ranking_df = pd.DataFrame({
    'Feature': X_train_sm.columns,
    'Rank': final_rfe.ranking_,
    'Selected': final_rfe.support_
}).sort_values(by='Rank')

print(ranking_df)

<hr style="border:10px solid gray">
<hr style="border:10px solid yellow">
<hr style="border:10px solid gray">